# 7. Leak-Free Downstream Evaluation

**Pipeline per fold (no data leakage):**
1. Split → train / test (from `Split_seed_0..9`)
2. cVAE batch correction: fit on **train only** → transform both
3. PCA: fit on **train only** → transform both
4. ML model: train on train → predict on test → metrics

**Models:** TabPFN (no inner CV), LogReg + LGBM (with inner GridSearchCV)

**Cohorts:** KIRC, NSCLC, SCAN-B

**Comparison:** `correction=none` vs `correction=cvae_adv2`

## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q lightgbm tabpfn
!pip install -q --upgrade ipython ipykernel


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY, SCVI_LATENT_DIM,
)
from batchcor_rna_emb.data_io import load_cohort
from batchcor_rna_emb.modeling.fold_runner import FoldConfig, run_experiment

set_logger(level="INFO")
logger.info("Imports OK")


## 1. Configuration

In [ ]:
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')

COHORTS = [
    "PUB_KIRC_ICI_combined",
    "NSCLC_Tissue_ICI_Pred",
    "PUB_BRCA_SCANB",
]

TARGETS = ["Response"]  # Add "Target_OS_bin", "Target_PFS_bin" later
MODELS = ["logreg", "lgbm"]  # Add "tabpfn" if installed
N_SPLITS = 10
CVAE_EPOCHS = 100

logger.info("Cohorts: {}", COHORTS)
logger.info("Targets: {}", TARGETS)
logger.info("Models: {}", MODELS)


## 2. Run Experiments
For each cohort × target × correction method, run all folds × all models.

In [ ]:
all_dfs = []

for cohort_name in COHORTS:
    zarr_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    
    logger.info("\n" + "=" * 70)
    logger.info("Loading cohort: {}", cohort_name)
    
    try:
        adata = load_cohort(zarr_path)
    except Exception as e:
        logger.warning("Failed to load {}: {}", cohort_name, e)
        continue
    
    logger.info("Loaded: {} samples", adata.n_obs)
    
    for target_col in TARGETS:
        if target_col not in adata.obs.columns:
            logger.warning("Target '{}' not found in {}, skipping", target_col, cohort_name)
            continue
        
        for correction in ["none", "cvae_adv2"]:
            logger.info("\n--- {} | target={} | correction={} ---", cohort_name, target_col, correction)
            
            cfg = FoldConfig(
                embedding_key=COMPASS_PT_EMBEDDING_KEY,
                target_col=target_col,
                correction_method=correction,
                n_pca=128,
                cvae_epochs=CVAE_EPOCHS,
                seed=SEED,
            )
            
            df = run_experiment(
                adata, cfg,
                model_names=MODELS,
                n_splits=N_SPLITS,
            )
            df["cohort"] = cohort_name
            all_dfs.append(df)

results = pd.concat(all_dfs, ignore_index=True)
logger.info("\nTotal results: {} rows", len(results))
display(results.head(20))


## 3. Summary: Mean ± Std across 10 Seeds

In [ ]:
# Aggregate metrics
metric_cols = ["roc_auc", "pr_auc", "f1", "balanced_accuracy"]
group_cols = ["cohort", "target", "correction", "model"]

summary = results.groupby(group_cols)[metric_cols].agg(["mean", "std"]).round(4)
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()

display(summary)


## 4. Visualization: ROC-AUC Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, len(COHORTS), figsize=(6 * len(COHORTS), 5), sharey=True)
if len(COHORTS) == 1:
    axes = [axes]

for ax, cohort in zip(axes, COHORTS):
    df_c = results[results["cohort"] == cohort]
    if df_c.empty:
        continue
    
    sns.boxplot(
        data=df_c, x="model", y="roc_auc", hue="correction",
        palette="Set2", ax=ax
    )
    ax.set_title(cohort.replace("_", " "), fontsize=12)
    ax.set_ylabel("ROC-AUC" if ax == axes[0] else "")
    ax.set_xlabel("Model")
    ax.legend(title="Correction", fontsize=8)
    ax.set_ylim(0.3, 1.0)

plt.suptitle("ROC-AUC: Raw vs Adv2-cVAE (10 seeds)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 5. Save Results

In [ ]:
output_dir = Path('/content/drive/MyDrive/data/results')
output_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(output_dir / "leak_free_evaluation_results.csv", index=False)
summary.to_csv(output_dir / "leak_free_evaluation_summary.csv", index=False)

logger.info("Results saved to {}", output_dir)
